In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks
from scipy.ndimage import label as ndlabel

import mlp_chinese

mlp_chinese.setup_chinese()

In [ ]:


ORIG_DIR = Path("deskew_output/original")
DPI = 300

image_paths = sorted(ORIG_DIR.glob("*.png"))
print(f"共发现 {len(image_paths)} 张图片：")
for p in image_paths:
    img = cv2.imread(str(p))
    h, w = img.shape[:2]
    print(f"{p.name}: {w}x{h}, ratio={w/h:.3f}, dtype={img.dtype}")

In [ ]:
sample_idx = 0   # 手动切页
SAMPLE_PATH = image_paths[sample_idx]

# bcompare_idx = 1 if len(image_paths) > 1 else 0

print(f"选取样本: {SAMPLE_PATH.name}")

img_bgr = cv2.imread(str(SAMPLE_PATH))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H_IMG, W_IMG = img_bgr.shape[:2]
print(f"尺寸: {W_IMG}x{H_IMG}")

fig, ax = plt.subplots(1, 1, figsize=(8, 9))
ax.imshow(img_rgb)
ax.set_title(f"原始图片: {SAMPLE_PATH.name}")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2Lab)

clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(16, 16))
img_clahe = clahe.apply(img_gray)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes[0, 0].imshow(img_rgb);                            axes[0, 0].set_title("RGB")
axes[0, 1].imshow(img_gray, cmap='gray');               axes[0, 1].set_title("Grayscale")
axes[0, 2].imshow(img_clahe, cmap='gray');              axes[0, 2].set_title("CLAHE")
axes[1, 0].imshow(img_hsv[:, :, 0], cmap='hsv');       axes[1, 0].set_title("HSV - H")
axes[1, 1].imshow(img_hsv[:, :, 1], cmap='gray');      axes[1, 1].set_title("HSV - S")
axes[1, 2].imshow(img_lab[:, :, 2], cmap='coolwarm');  axes[1, 2].set_title("Lab - a")
axes[1, 2].set_title("Lab - b (blue<128)")

gx = cv2.Sobel(img_gray, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(img_gray, cv2.CV_32F, 0, 1, ksize=3)
grad_mag = cv2.magnitude(gx, gy)

for ax in axes.flat:
    ax.axis("off")
plt.suptitle("预处理: 多颜色空间", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
h_ch, s_ch, v_ch = img_hsv[:, :, 0], img_hsv[:, :, 1], img_hsv[:, :, 2]

H_LOW, H_HIGH = 85, 130
h_mask = (h_ch >= H_LOW) & (h_ch <= H_HIGH)

# Otsu 自适应饱和度阈值
s_in_hue = s_ch[h_mask]
if len(s_in_hue) > 100:
    otsu_thresh, _ = cv2.threshold(s_in_hue, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    S_THRESH = max(int(otsu_thresh * 0.6), 20)
else:
    S_THRESH = 30
print(f"Otsu={otsu_thresh:.0f}, 使用 S 阈值: {S_THRESH}")

color_mask = h_mask & (s_ch >= S_THRESH)
Rc = np.zeros_like(img_gray, dtype=np.float32)
Rc[color_mask] = s_ch[color_mask].astype(np.float32) / 255.0

print(f"颜色 mask 像素占比: {color_mask.sum() / color_mask.size * 100:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(h_mask, cmap='gray');    axes[0].set_title(f"H in [{H_LOW}, {H_HIGH}]")
axes[1].imshow(color_mask, cmap='gray'); axes[1].set_title(f"H + S>={S_THRESH}")
axes[2].imshow(Rc, cmap='hot');          axes[2].set_title("Rc (颜色响应)")
for ax in axes:
    ax.axis("off")
plt.suptitle("颜色支路: HSV 蓝色提取", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
sobel_x = cv2.Sobel(img_clahe, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img_clahe, cv2.CV_64F, 0, 1, ksize=3)
abs_gx = np.abs(sobel_x)
abs_gy = np.abs(sobel_y)

eps = 1e-6
# 水平线: y 方向梯度强且 x 方向弱 → gy^2/(gx+gy)
h_line_resp = abs_gy / (abs_gx + abs_gy + eps) * abs_gy
# 竖直线: x 方向梯度强且 y 方向弱 → gx^2/(gx+gy)
v_line_resp = abs_gx / (abs_gx + abs_gy + eps) * abs_gx

Rg_raw = h_line_resp + v_line_resp
Rg = (Rg_raw / Rg_raw.max()).astype(np.float32) if Rg_raw.max() > 0 else Rg_raw.astype(np.float32)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(abs_gx, cmap='gray', vmax=np.percentile(abs_gx, 99))
axes[0].set_title("|Sobel X| (竖直边缘)")
axes[1].imshow(abs_gy, cmap='gray', vmax=np.percentile(abs_gy, 99))
axes[1].set_title("|Sobel Y| (水平边缘)")
axes[2].imshow(h_line_resp, cmap='hot', vmax=np.percentile(h_line_resp, 99))
axes[2].set_title("水平线响应")
axes[3].imshow(Rg, cmap='hot', vmax=np.percentile(Rg, 99))
axes[3].set_title("Rg (梯度响应)")
for ax in axes:
    ax.axis("off")
plt.suptitle("梯度支路: Sobel 方向梯度", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
mask_float = color_mask.astype(np.float32)
row_proj = mask_float.mean(axis=1)  # 水平投影 -> 横线
col_proj = mask_float.mean(axis=0)  # 垂直投影 -> 竖线

def estimate_spacing_fft(projection, min_period=30, max_period=800):
    """FFT 估算投影信号主周期（像素间距）。"""
    sig = projection - projection.mean()
    n = len(sig)
    fft_mag = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(n)
    valid = (freqs > 1.0 / max_period) & (freqs < 1.0 / min_period) & (freqs > 0)
    if not valid.any():
        return None, fft_mag, freqs
    fft_valid = fft_mag.copy()
    fft_valid[~valid] = 0
    peak_idx = np.argmax(fft_valid)
    peak_freq = freqs[peak_idx]
    spacing = 1.0 / peak_freq if peak_freq > 0 else None
    return spacing, fft_mag, freqs

h_spacing, h_fft, h_freqs = estimate_spacing_fft(row_proj)
v_spacing, v_fft, v_freqs = estimate_spacing_fft(col_proj)

if h_spacing:
    print(f"FFT 横线主周期: {h_spacing:.1f} px ({h_spacing/DPI*25.4:.1f} mm)")
else:
    print("横线间距估算失败")
if v_spacing:
    print(f"FFT 竖线主周期: {v_spacing:.1f} px ({v_spacing/DPI*25.4:.1f} mm)")
else:
    print("竖线间距估算失败")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(row_proj, linewidth=0.3)
axes[0, 0].set_title("水平投影 (行求和)")
axes[0, 0].set_xlabel("y")

valid_h = (h_freqs > 0) & (h_freqs < 0.05)
if valid_h.any():
    axes[0, 1].plot(1.0 / h_freqs[valid_h], h_fft[valid_h], linewidth=0.5)
    axes[0, 1].set_title(f"FFT (水平) → 主周期 {h_spacing:.0f}px" if h_spacing else "FFT (水平)")
    axes[0, 1].set_xlabel("周期 (px)")
    if h_spacing:
        axes[0, 1].axvline(h_spacing, color='r', ls='--', alpha=0.7)

axes[1, 0].plot(col_proj, linewidth=0.3)
axes[1, 0].set_title("垂直投影 (列求和)")
axes[1, 0].set_xlabel("x")

valid_v = (v_freqs > 0) & (v_freqs < 0.05)
if valid_v.any():
    axes[1, 1].plot(1.0 / v_freqs[valid_v], v_fft[valid_v], linewidth=0.5)
    axes[1, 1].set_title(f"FFT (垂直) → 主周期 {v_spacing:.0f}px" if v_spacing else "FFT (垂直)")
    axes[1, 1].set_xlabel("周期 (px)")
    if v_spacing:
        axes[1, 1].axvline(v_spacing, color='r', ls='--', alpha=0.7)

plt.suptitle("周期性支路: 投影 + FFT", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
WC, WG = 0.7, 0.3
R_fused = WC * Rc + WG * Rg
R_fused = R_fused / R_fused.max() if R_fused.max() > 0 else R_fused

R_u8 = (R_fused * 255).astype(np.uint8)
_, R_bin = cv2.threshold(R_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 方向闭运算：分别用水平和竖直核连接断裂线段
kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1))
kernel_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15))
closed_h = cv2.morphologyEx(R_bin, cv2.MORPH_CLOSE, kernel_h)
closed_v = cv2.morphologyEx(R_bin, cv2.MORPH_CLOSE, kernel_v)
R_closed = cv2.bitwise_or(closed_h, closed_v)

# 开运算去噪点
R_clean = cv2.morphologyEx(R_closed, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

# 去除面积过小的连通域
MIN_AREA = 200
labels_arr, n_labels = ndlabel(R_clean)
for i in range(1, n_labels + 1):
    if (labels_arr == i).sum() < MIN_AREA:
        R_clean[labels_arr == i] = 0

grid_mask = R_clean
print(f"融合 mask 像素占比: {(grid_mask > 0).sum() / grid_mask.size * 100:.2f}%")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(R_fused, cmap='hot');  axes[0].set_title("融合响应 R")
axes[1].imshow(R_bin, cmap='gray');   axes[1].set_title("Otsu 二值化")
axes[2].imshow(R_closed, cmap='gray'); axes[2].set_title("闭运算连接")
axes[3].imshow(grid_mask, cmap='gray'); axes[3].set_title("最终 grid_mask")
for ax in axes:
    ax.axis("off")
plt.suptitle("多支路融合 + 形态学清理", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
edges = cv2.Canny(grid_mask, 50, 150)

lines_raw = cv2.HoughLinesP(
    edges, rho=1, theta=np.pi / 180, threshold=100,
    minLineLength=W_IMG // 8, maxLineGap=30,
)
if lines_raw is None:
    print("阈值=100 未检测到线段, 降低到 50 重试...")
    lines_raw = cv2.HoughLinesP(
        edges, 1, np.pi / 180, 50,
        minLineLength=W_IMG // 16, maxLineGap=50,
    )

lines = lines_raw[:, 0, :] if lines_raw is not None else np.empty((0, 4))
print(f"检测到 {len(lines)} 条线段")

angles = np.array([np.degrees(np.arctan2(l[3]-l[1], l[2]-l[0])) for l in lines])
lengths = np.array([np.hypot(l[2]-l[0], l[3]-l[1]) for l in lines])

ANGLE_TOL = 15
h_idx = np.abs(angles) < ANGLE_TOL
v_idx = (np.abs(angles - 90) < ANGLE_TOL) | (np.abs(angles + 90) < ANGLE_TOL)

h_lines_raw = lines[h_idx]
v_lines_raw = lines[v_idx]
h_angles = angles[h_idx]
v_angles = angles[v_idx]

print(f"水平线: {len(h_lines_raw)} 条, 中位角度: {np.median(h_angles):.2f}°" if len(h_angles) > 0 else "水平线: 0 条")
print(f"竖直线: {len(v_lines_raw)} 条, 中位角度: {np.median(v_angles):.2f}°" if len(v_angles) > 0 else "竖直线: 0 条")

vis = img_rgb.copy()
for x1, y1, x2, y2 in h_lines_raw:
    cv2.line(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
for x1, y1, x2, y2 in v_lines_raw:
    cv2.line(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
axes[0].hist(angles, bins=180, range=(-90, 90), edgecolor='none')
axes[0].axvline(0, color='r', ls='--', label='水平')
axes[0].axvline(90, color='b', ls='--', label='竖直')
axes[0].axvline(-90, color='b', ls='--')
axes[0].set_title("线段方向角分布")
axes[0].set_xlabel("角度 (°)")
axes[0].legend()

axes[1].imshow(vis)
axes[1].set_title(f"线段检测: 红=水平({len(h_lines_raw)}), 蓝=竖直({len(v_lines_raw)})")
axes[1].axis("off")
plt.suptitle("线段检测 + 方向聚类", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def merge_collinear_lines(lines, is_horizontal=True, pos_tol=15):
    """
    将位置接近的平行线段合并为代表线。
    水平线按 y 聚类，竖直线按 x 聚类。
    """
    if len(lines) == 0:
        return np.empty((0, 4), dtype=int), np.array([])

    if is_horizontal:
        positions = (lines[:, 1] + lines[:, 3]) / 2.0
    else:
        positions = (lines[:, 0] + lines[:, 2]) / 2.0

    order = np.argsort(positions)
    sorted_lines = lines[order]
    sorted_pos = positions[order]

    groups = []
    current = [0]
    for i in range(1, len(sorted_pos)):
        if sorted_pos[i] - sorted_pos[current[-1]] <= pos_tol:
            current.append(i)
        else:
            groups.append(current)
            current = [i]
    groups.append(current)

    merged, mpos = [], []
    for g in groups:
        gl = sorted_lines[g]
        if is_horizontal:
            y_avg = int(np.mean(gl[:, [1, 3]]))
            x_min = int(gl[:, [0, 2]].min())
            x_max = int(gl[:, [0, 2]].max())
            merged.append([x_min, y_avg, x_max, y_avg])
            mpos.append(y_avg)
        else:
            x_avg = int(np.mean(gl[:, [0, 2]]))
            y_min = int(gl[:, [1, 3]].min())
            y_max = int(gl[:, [1, 3]].max())
            merged.append([x_avg, y_min, x_avg, y_max])
            mpos.append(x_avg)

    return np.array(merged, dtype=int), np.array(mpos, dtype=float)

In [ ]:
h_merged, h_positions = merge_collinear_lines(h_lines_raw, is_horizontal=True, pos_tol=15)
v_merged, v_positions = merge_collinear_lines(v_lines_raw, is_horizontal=False, pos_tol=15)

print(f"合并后水平线: {len(h_merged)} 条")
print(f"合并后竖直线: {len(v_merged)} 条")

h_spacings = np.diff(h_positions) if len(h_positions) > 1 else np.array([])
v_spacings = np.diff(v_positions) if len(v_positions) > 1 else np.array([])

if len(h_spacings) > 0:
    print(f"\n横线间距: mean={h_spacings.mean():.1f}, median={np.median(h_spacings):.1f}, "
          f"std={h_spacings.std():.1f}, CV={h_spacings.std()/h_spacings.mean():.3f}")
    print(f"横线间距序列 (前20): {h_spacings[:20].astype(int)}")

if len(v_spacings) > 0:
    print(f"\n竖线间距: mean={v_spacings.mean():.1f}, median={np.median(v_spacings):.1f}, "
          f"std={v_spacings.std():.1f}, CV={v_spacings.std()/v_spacings.mean():.3f}")
    print(f"竖线间距序列 (前20): {v_spacings[:20].astype(int)}")

def detect_tianzige_pattern(spacings, ratio_range=(1.5, 3.0)):
    """检测间距是否呈现田字格的大-小交替模式。"""
    if len(spacings) < 4:
        return False, None, None
    med = np.median(spacings)
    big = spacings[spacings > med]
    small = spacings[spacings <= med]
    if len(big) == 0 or len(small) == 0:
        return False, med, None
    ratio = np.median(big) / np.median(small)
    return ratio_range[0] <= ratio <= ratio_range[1], np.median(small), np.median(big)

is_tz_h, small_h, big_h = detect_tianzige_pattern(h_spacings)
is_tz_v, small_v, big_v = detect_tianzige_pattern(v_spacings)

print(f"\n田字格模式 (横): {'是' if is_tz_h else '否'}")
if is_tz_h and small_h and big_h:
    print(f"  内线={small_h:.0f}px, 外框={big_h:.0f}px, 比值={big_h/small_h:.2f}")
print(f"田字格模式 (竖): {'是' if is_tz_v else '否'}")
if is_tz_v and small_v and big_v:
    print(f"  内线={small_v:.0f}px, 外框={big_v:.0f}px, 比值={big_v/small_v:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
if len(h_spacings) > 0:
    axes[0].bar(range(len(h_spacings)), h_spacings, color='coral')
    axes[0].axhline(np.median(h_spacings), color='k', ls='--',
                     label=f'median={np.median(h_spacings):.0f}')
    axes[0].set_title("横线间距序列")
    axes[0].set_ylabel("间距 (px)")
    axes[0].legend()
if len(v_spacings) > 0:
    axes[1].bar(range(len(v_spacings)), v_spacings, color='steelblue')
    axes[1].axhline(np.median(v_spacings), color='k', ls='--',
                     label=f'median={np.median(v_spacings):.0f}')
    axes[1].set_title("竖线间距序列")
    axes[1].set_ylabel("间距 (px)")
    axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
def compute_intersections(h_merged, v_merged):
    """水平线 x 竖直线 → 交点矩阵 (n_h, n_v, 2)。"""
    n_h, n_v = len(h_merged), len(v_merged)
    if n_h == 0 or n_v == 0:
        return np.empty((0, 0, 2))
    pts = np.zeros((n_h, n_v, 2), dtype=np.float32)
    for i, hl in enumerate(h_merged):
        y = (hl[1] + hl[3]) / 2.0
        for j, vl in enumerate(v_merged):
            x = (vl[0] + vl[2]) / 2.0
            pts[i, j] = [x, y]
    return pts

intersections = compute_intersections(h_merged, v_merged)
n_rows_g, n_cols_g = intersections.shape[:2]
print(f"交点矩阵: {n_rows_g} x {n_cols_g} = {n_rows_g * n_cols_g} 个交点")

# 仿射网格拟合 + 残差
residuals = None
if n_rows_g >= 2 and n_cols_g >= 2:
    pts_flat = intersections.reshape(-1, 2)
    row_idx, col_idx = np.meshgrid(np.arange(n_rows_g), np.arange(n_cols_g), indexing='ij')
    rc_mat = np.column_stack([col_idx.ravel(), row_idx.ravel(), np.ones(len(pts_flat))])

    Ax, _, _, _ = np.linalg.lstsq(rc_mat, pts_flat[:, 0], rcond=None)
    Ay, _, _, _ = np.linalg.lstsq(rc_mat, pts_flat[:, 1], rcond=None)

    ideal_pts = np.column_stack([rc_mat @ Ax, rc_mat @ Ay])
    residuals = np.linalg.norm(pts_flat - ideal_pts, axis=1)

    print(f"\n仿射拟合残差: mean={residuals.mean():.2f}px, max={residuals.max():.2f}px")
    print(f"拟合列间距≈{Ax[0]:.1f}px, 行间距≈{Ay[1]:.1f}px")
    skew_angle = np.degrees(np.arctan2(Ax[1], Ax[0]))
    print(f"网格倾角: {skew_angle:.3f}°")

# 可视化
vis2 = img_rgb.copy()
if intersections.size > 0:
    for pt in intersections.reshape(-1, 2):
        cv2.circle(vis2, (int(pt[0]), int(pt[1])), 8, (255, 0, 0), -1)

fig, ax = plt.subplots(1, 1, figsize=(10, 11))
ax.imshow(vis2)
ax.set_title(f"交点: {n_rows_g} x {n_cols_g} = {n_rows_g * n_cols_g} 个")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def evaluate_quality(h_pos, v_pos, h_angs, v_angs, inters, img_shape, res=None):
    """综合质量评估，返回各项分数和总置信度。"""
    scores = {}

    # S_dir: 方向集中度
    s_dir_h = max(0, 1.0 - np.std(h_angs) / 5.0) if len(h_angs) > 0 else 0.0
    s_dir_v = max(0, 1.0 - np.std(v_angs) / 5.0) if len(v_angs) > 0 else 0.0
    scores['S_dir'] = (s_dir_h + s_dir_v) / 2.0

    # S_spacing: 间距稳定性
    cvs = []
    h_sp = np.diff(h_pos) if len(h_pos) > 1 else np.array([])
    v_sp = np.diff(v_pos) if len(v_pos) > 1 else np.array([])
    if len(h_sp) > 1:
        cvs.append(h_sp.std() / h_sp.mean())
    if len(v_sp) > 1:
        cvs.append(v_sp.std() / v_sp.mean())
    scores['S_spacing'] = max(0, 1.0 - np.mean(cvs) * 2) if cvs else 0.0

    # S_coverage: 网格覆盖率
    if len(h_pos) >= 2 and len(v_pos) >= 2:
        grid_area = (h_pos[-1] - h_pos[0]) * (v_pos[-1] - v_pos[0])
        coverage = grid_area / (img_shape[0] * img_shape[1])
        scores['S_coverage'] = min(1.0, coverage / 0.5)
    else:
        scores['S_coverage'] = 0.0

    # S_residual: 拟合残差
    if res is not None and len(res) > 0:
        scores['S_residual'] = max(0, 1.0 - res.mean() / 20.0)
    else:
        scores['S_residual'] = 0.5

    # 综合
    scores['confidence'] = (
        0.2 * scores['S_dir'] +
        0.3 * scores['S_spacing'] +
        0.2 * scores['S_coverage'] +
        0.3 * scores['S_residual']
    )
    return scores

quality = evaluate_quality(h_positions, v_positions, h_angles, v_angles,
                           intersections, img_bgr.shape[:2], residuals)

print("=== 质量评估 ===")
for k, v in quality.items():
    print(f"  {k}: {v:.4f}")
print(f"\n>>> 总置信度: {quality['confidence']:.3f}")

In [ ]:
def visualize_grid_result(img_rgb, h_merged, v_merged, intersections, title=""):
    """原图上叠加检测结果。"""
    vis = img_rgb.copy()
    for x1, y1, x2, y2 in h_merged:
        cv2.line(vis, (x1, y1), (x2, y2), (255, 50, 50), 3)
    for x1, y1, x2, y2 in v_merged:
        cv2.line(vis, (x1, y1), (x2, y2), (50, 50, 255), 3)

    if intersections.size > 0:
        for pt in intersections.reshape(-1, 2):
            cv2.circle(vis, (int(pt[0]), int(pt[1])), 10, (0, 255, 0), -1)

    n_r, n_c = intersections.shape[:2]
    cell_count = 0
    for i in range(n_r - 1):
        for j in range(n_c - 1):
            tl = intersections[i, j].astype(int)
            br = intersections[i + 1, j + 1].astype(int)
            cv2.rectangle(vis, tuple(tl), tuple(br), (255, 200, 0), 2)
            cell_count += 1

    fig, axes = plt.subplots(1, 2, figsize=(18, 10))
    axes[0].imshow(img_rgb)
    axes[0].set_title("原始图片")
    axes[0].axis("off")
    axes[1].imshow(vis)
    axes[1].set_title(f"{title}\n红=横线({len(h_merged)}), 蓝=竖线({len(v_merged)}), "
                      f"绿=交点({n_r*n_c}), 黄=单元格({cell_count})")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    return cell_count

visualize_grid_result(img_rgb, h_merged, v_merged, intersections,
                      title=f"{SAMPLE_PATH.name} — 置信度 {quality['confidence']:.3f}")

In [ ]:
def detect_grid(img_path, h_range=(85, 130), angle_tol=15, pos_tol=15,
                wc=0.7, wg=0.3, min_area=200):
    """
    完整田字格网格检测 pipeline。
    """
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h_img, w_img = img_bgr.shape[:2]
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    cl = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(16, 16))
    img_cl = cl.apply(img_gray)

    # --- 颜色支路 ---
    hc, sc = img_hsv[:, :, 0], img_hsv[:, :, 1]
    hmask = (hc >= h_range[0]) & (hc <= h_range[1])
    s_vals = sc[hmask]
    if len(s_vals) > 100:
        ot, _ = cv2.threshold(s_vals, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        st = max(int(ot * 0.6), 20)
    else:
        st = 30
    cmask = hmask & (sc >= st)
    Rc = np.zeros_like(img_gray, dtype=np.float32)
    Rc[cmask] = sc[cmask].astype(np.float32) / 255.0

    # --- 梯度支路 ---
    gx = np.abs(cv2.Sobel(img_cl, cv2.CV_64F, 1, 0, ksize=3))
    gy = np.abs(cv2.Sobel(img_cl, cv2.CV_64F, 0, 1, ksize=3))
    eps = 1e-6
    Rg_raw = gy / (gx + gy + eps) * gy + gx / (gx + gy + eps) * gx
    Rg = (Rg_raw / Rg_raw.max()).astype(np.float32) if Rg_raw.max() > 0 else Rg_raw.astype(np.float32)

    # --- 融合 ---
    R = wc * Rc + wg * Rg
    R = R / R.max() if R.max() > 0 else R
    Ru8 = (R * 255).astype(np.uint8)
    _, Rb = cv2.threshold(Ru8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # --- 形态学 ---
    kh = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1))
    kv = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15))
    closed = cv2.bitwise_or(
        cv2.morphologyEx(Rb, cv2.MORPH_CLOSE, kh),
        cv2.morphologyEx(Rb, cv2.MORPH_CLOSE, kv),
    )
    cleaned = cv2.morphologyEx(closed, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    la, nl = ndlabel(cleaned)
    for i in range(1, nl + 1):
        if (la == i).sum() < min_area:
            cleaned[la == i] = 0

    # --- 线段检测 ---
    ed = cv2.Canny(cleaned, 50, 150)
    raw = cv2.HoughLinesP(ed, 1, np.pi / 180, 100, minLineLength=w_img // 8, maxLineGap=30)
    if raw is None:
        raw = cv2.HoughLinesP(ed, 1, np.pi / 180, 50, minLineLength=w_img // 16, maxLineGap=50)
    segs = raw[:, 0, :] if raw is not None else np.empty((0, 4))

    angs = np.array([np.degrees(np.arctan2(s[3]-s[1], s[2]-s[0])) for s in segs])
    hi = np.abs(angs) < angle_tol
    vi = (np.abs(angs - 90) < angle_tol) | (np.abs(angs + 90) < angle_tol)

    hm, hp = merge_collinear_lines(segs[hi], True, pos_tol)
    vm, vp = merge_collinear_lines(segs[vi], False, pos_tol)

    inters = compute_intersections(hm, vm)

    # 拟合残差
    res = None
    if inters.shape[0] >= 2 and inters.shape[1] >= 2:
        pf = inters.reshape(-1, 2)
        ri, ci = np.meshgrid(np.arange(inters.shape[0]), np.arange(inters.shape[1]), indexing='ij')
        rcm = np.column_stack([ci.ravel(), ri.ravel(), np.ones(len(pf))])
        ax_, _, _, _ = np.linalg.lstsq(rcm, pf[:, 0], rcond=None)
        ay_, _, _, _ = np.linalg.lstsq(rcm, pf[:, 1], rcond=None)
        ideal = np.column_stack([rcm @ ax_, rcm @ ay_])
        res = np.linalg.norm(pf - ideal, axis=1)

    qual = evaluate_quality(hp, vp, angs[hi], angs[vi], inters, (h_img, w_img), res)

    return {
        'img_rgb': img_rgb,
        'grid_mask': cleaned,
        'h_lines': hm, 'v_lines': vm,
        'h_positions': hp, 'v_positions': vp,
        'intersections': inters,
        'quality': qual,
        'residuals': res,
    }

In [ ]:
results = {}
for p in image_paths:
    print(f"\n{'='*60}")
    print(f"处理: {p.name}")
    r = detect_grid(p)
    results[p.name] = r
    q = r['quality']
    sh = r['intersections'].shape
    print(f"  水平线: {len(r['h_lines'])}, 竖直线: {len(r['v_lines'])}")
    print(f"  交点: {sh[0]}x{sh[1]}")
    print(f"  置信度: {q['confidence']:.3f}")

print(f"\n{'='*60}")
print(f"\n{'页面':<20} {'水平线':>6} {'竖直线':>6} {'交点':>10} {'置信度':>8}")
print("-" * 55)
for name, r in results.items():
    q = r['quality']
    sh = r['intersections'].shape
    print(f"{name:<20} {len(r['h_lines']):>6} {len(r['v_lines']):>6} "
          f"{sh[0]:>4}x{sh[1]:<4} {q['confidence']:>8.3f}")